# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
# 1. Cleaning
#drop id and name columns
spaceship.drop(columns = ['PassengerId', 'Name'], inplace = True)
#drop na
spaceship.dropna(inplace = True)


## Feature Scaling

In [12]:
#Scaling
#Drop null values
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
spaceship.dropna(inplace = True)

#dummification

spaceship_cat = spaceship.select_dtypes('object')
spaceship_num = spaceship.select_dtypes('number')
# Dummisification = transformin nominal columns to num columns
#Create OneHotEncoder class instance
ohe = OneHotEncoder(sparse_output=False, drop = "first")

#find the unique values for every column
ohe.fit(spaceship_cat)

#output np array
spaceship_cat_np = ohe.transform(spaceship_cat)


#transform np array to data Frame
spaceship_cat_df = pd.DataFrame(spaceship_cat_np, columns = ohe.get_feature_names_out(), index = space_ship_cat.index)

#Scaling the num df using stdandard scaler
#create instance
scaler = StandardScaler()

#fit the scaler
scaler.fit(spaceship_num)
#Transform scaler
spaceship_num_np = scaler.transform(spaceship_num)

#convert back the np array to df
spaceship_num_df = pd.DataFrame(spaceship_num_np, columns = spaceship_num.columns, index = space_ship_num.index)

spaceship_scaled = pd.concat([spaceship_cat_df, spaceship_num_df], axis = 1 )
spaceship_scaled.head()

,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_A/0/S,Cabin_A/1/S,Cabin_A/10/P,Cabin_A/10/S,Cabin_A/100/S,Cabin_A/101/S,Cabin_A/102/S,...,Cabin_T/3/P,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.695365,-0.346316,-0.286103,-0.282915,-0.275577,-0.270290
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,-0.337089,-0.178108,-0.280735,-0.243729,0.206465,-0.231242
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,2.003140,-0.279959,1.846533,-0.282915,5.620436,-0.226804
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.282383,-0.346316,0.479046,0.298603,2.647405,-0.099010
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,-0.887732,0.121271,-0.244356,-0.046233,0.220513,-0.268515


## Feature Selection
Features to keep: Higher correlation with the target column
After checking the correlation between feature columns, we found that the spearman/pearson correlation is not high, so we keep the following features: 
('HomePlanet_Europa', 'CryoSleep_True', 'Destination_TRAPPIST-1e', 'RoomService', 'Spa', 'VRDeck')


In [13]:
target = spaceship['Transported']
#Check correlation between features and target col

correlation_feat_taget = spaceship_scaled.corrwith(target)
#correlation_feat_target
#drop columns with low correlation with target column
feat_col_to_keep = [row for row, value in correlation_feat_taget.items() if abs(value) > 0.1]

  
#correlation_feat_taget
spaceship_scaled_features = spaceship_scaled[feat_col_to_keep]
print("columns to keep:\n",feat_col_to_keep )

#Check correlation between features
correlation_feat_feat_spearman =spaceship_scaled_features.corr(method = "spearman")
correlation_feat_feat_pearson =spaceship_scaled_features.corr(method = "pearson")

columns to keep:
 ['HomePlanet_Europa', 'CryoSleep_True', 'Destination_TRAPPIST-1e', 'RoomService', 'Spa', 'VRDeck']


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

##  how can we improve model when we fine tune it's hyperparameters?
From last lab, I concluded that the best model is the Bagging and Pasting model

- Evaluate your model

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import classification_report

#Perform Train Test Split
X_train, X_test, y_train, y_test = train_test_split(spaceship_scaled_features, target, test_size = 0.2, random_state = 0)

#Create model class
class_bag = BaggingClassifier(DecisionTreeClassifier(max_depth = 20),
                                   max_samples = 1000
                                  )
    
#train the model
class_bag.fit(X_train, y_train)

#Evaluate mode lperformance
y_pred_test = class_bag.predict(X_test)

print("Calssification Report:\n ", classification_report(y_test, y_pred_test))

Calssification Report:
                precision    recall  f1-score   support

       False       0.84      0.73      0.78       670
        True       0.77      0.86      0.81       683

    accuracy                           0.80      1353
   macro avg       0.80      0.80      0.79      1353
weighted avg       0.80      0.80      0.80      1353



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [27]:
from sklearn.model_selection import GridSearchCV
import time

#create the instance of our model
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    random_state=42
)

#Parameters grid
param_grid = {
    'n_estimators': [50, 100],
    'max_samples': [500, 1000],
    'max_features': [0.7, 1.0],
    'estimator__max_depth': [10, 20]
    
}



- Run Grid Search

In [28]:
# We need to set this two variables to be able to compute a confidence interval
confidence_level = 0.95
folds = 5

#Create an instance of GridSearchCV
grid = GridSearchCV(bag, param_grid = param_grid, cv=folds, verbose=10)

start_time = time.time()
#Train GridSearchCV
grid.fit(X_train, y_train)

end_time = time.time()



Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV 1/5; 1/16] START estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50
[CV 1/5; 1/16] END estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50;, score=0.773 total time=   0.2s
[CV 2/5; 1/16] START estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50
[CV 2/5; 1/16] END estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50;, score=0.749 total time=   0.2s
[CV 3/5; 1/16] START estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50
[CV 3/5; 1/16] END estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50;, score=0.770 total time=   0.2s
[CV 4/5; 1/16] START estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50
[CV 4/5; 1/16] END estimator__max_depth=10, max_features=0.7, max_samples=500, n_estimators=50;, score=0.783 total time=   0.5s
[CV 5/5; 1/16] START estimator__max_dep

- Evaluate your model

In [29]:
print("\n")
print(f"Training time of Baggin model is: {end_time - start_time: .2f} seconds")
print("\n")

print(f"The best combination of hyperparameters has been: {grid.best_params_}")
print(f"The Accuracy is: {grid.best_score_: .4f}")



Training time of Baggin model is:  48.09 seconds


The best combination of hyperparameters has been: {'estimator__max_depth': 10, 'max_features': 1.0, 'max_samples': 500, 'n_estimators': 50}
The Accuracy is:  0.7849


# THE best hyperparameters are {'estimator__max_depth': 10, 'max_features': 1.0, 'max_samples': 500, 'n_estimators': 50} which gives accuracy=  0.7849.